In [ ]:
# ============================================================
# VALIDATION A — BRANCH A
# D6 — Microsoft FY24 Q1 Press Release
# ============================================================
#
# Methodology stage covered:
# Stage 4 — Post-processing and Validation
#
# Compares the preserved Branch A extraction against the
# fixed Stage 1 document-grounded reference dataset.
# ============================================================

from google.colab import files
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter

import hashlib
import json
import math
import re
import unicodedata

import pandas as pd
from scipy.optimize import linear_sum_assignment


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D6"
DOCUMENT_NAME = "Microsoft FY24 Q1 Press Release"

BRANCH = "A"
BRANCH_NAME = "Direct Ingestion"
INPUT_REPRESENTATION = "Original PDF"

EXPECTED_REFERENCE_RECORD_COUNT = 147
EXPECTED_EXTRACTION_RECORD_COUNT = 147

EXPECTED_CATEGORY_COUNTS = {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
}

FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit",
    "Reporting Period",
    "Source Location"
]

ALIGNMENT_IDENTITY_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Reporting Period",
    "Source Location"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit"
]

STRING_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Unit",
    "Reporting Period",
    "Source Location"
]

NUMERIC_FIELDS = [
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change"
]

BLOCK_FIELDS = [
    "Category",
    "Source Location"
]

MATCH_SCORE_THRESHOLD = 0.34

OUTPUT_DIR = Path("outputs_D6_validation_A")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Expected extraction records:", EXPECTED_EXTRACTION_RECORD_COUNT)
print("Primary correctness fields:", len(PRIMARY_CORRECTNESS_FIELDS))
print("Output directory:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Validation inputs
# ============================================================

print(
    "Upload exactly three files:\n"
    "1. D6_reference_values.csv\n"
    "2. D6_branch_A_combined_parsed_extraction.json\n"
    "3. D6_branch_A_technical_diagnostics.json"
)

uploaded = files.upload()

csv_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".csv")
]

json_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".json")
]

if len(csv_paths) != 1:
    raise ValueError("Upload exactly one CSV reference file.")

if len(json_paths) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the combined extraction "
        "and the Branch A technical diagnostics."
    )

REFERENCE_PATH = csv_paths[0]

technical_diagnostics_candidates = [
    path
    for path in json_paths
    if "technical_diagnostics" in path.name.lower()
]

extraction_candidates = [
    path
    for path in json_paths
    if "technical_diagnostics" not in path.name.lower()
]

if len(technical_diagnostics_candidates) != 1:
    raise ValueError(
        "Could not uniquely identify "
        "D6_branch_A_technical_diagnostics.json."
    )

TECHNICAL_DIAGNOSTICS_PATH = (
    technical_diagnostics_candidates[0]
)

EXTRACTION_PATH = extraction_candidates[0]

print("Reference:", REFERENCE_PATH.name)
print("Extraction:", EXTRACTION_PATH.name)
print(
    "Technical diagnostics:",
    TECHNICAL_DIAGNOSTICS_PATH.name
)

In [ ]:
# ============================================================
# 3. File hashing utility and input provenance
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
TECHNICAL_DIAGNOSTICS_SHA256 = sha256_file(TECHNICAL_DIAGNOSTICS_PATH)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print(
    "Technical diagnostics SHA-256:",
    TECHNICAL_DIAGNOSTICS_SHA256
)

In [ ]:
# ============================================================
# 4. Load fixed Stage 1 reference dataset
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)


def restore_csv_null(value):
    if value is None:
        return None

    if isinstance(value, str) and value == "":
        return None

    return value


def restore_mixed_number(value):
    if value is None or not isinstance(value, str):
        return value

    text = (
        value.strip()
        .replace(",", "")
        .replace("−", "-")
        .replace("–", "-")
    )

    if re.fullmatch(r"-?\d+", text):
        return int(text)

    if re.fullmatch(r"-?\d+\.\d+", text):
        return float(text)

    return value


for column in reference_df.columns:
    reference_df[column] = reference_df[column].map(restore_csv_null)

for field in NUMERIC_FIELDS:
    reference_df[field] = reference_df[field].map(restore_mixed_number)


print("Reference shape:", reference_df.shape)
print("Reference columns:", reference_df.columns.tolist())

display(reference_df.head(10))


In [ ]:
# ============================================================
# 5. Load untouched Branch A combined extraction
# ============================================================

with EXTRACTION_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    extraction_payload = json.load(file)

if (
    not isinstance(extraction_payload, dict)
    or extraction_payload.get("document_id") != DOCUMENT_ID
    or extraction_payload.get("branch") != BRANCH
    or not isinstance(
        extraction_payload.get("records"),
        list
    )
):
    raise ValueError(
        "The Branch A combined extraction has an "
        "unexpected top-level structure."
    )

extracted_records = extraction_payload["records"]

print(
    "Combined extracted records:",
    len(extracted_records)
)

with TECHNICAL_DIAGNOSTICS_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    branch_technical_diagnostics = json.load(file)

branch_diagnostics_document_id_correct = (
    branch_technical_diagnostics.get("document_id")
    == DOCUMENT_ID
)

branch_diagnostics_branch_correct = (
    branch_technical_diagnostics.get("branch")
    == BRANCH
)

branch_structurally_evaluable = bool(
    branch_technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

print(
    "Technical-diagnostics document ID correct:",
    branch_diagnostics_document_id_correct
)

print(
    "Technical-diagnostics branch correct:",
    branch_diagnostics_branch_correct
)

print(
    "Branch A structurally evaluable:",
    branch_structurally_evaluable
)


In [ ]:
# ============================================================
# 6. Preserve raw record structure and create comparison table
# ============================================================

raw_record_rows = []
schema_issue_rows = []

for record_index, record in enumerate(extracted_records):

    if not isinstance(record, dict):
        schema_issue_rows.append(
            {
                "Record Index": record_index,
                "Issue": "Record is not a JSON object",
                "Missing Fields": FIELDS,
                "Extra Fields": None,
                "Expected Fields": FIELDS,
                "Observed Fields": None
            }
        )

        comparison_record = {field: None for field in FIELDS}

    else:
        observed_fields = list(record.keys())

        missing_fields = [
            field
            for field in FIELDS
            if field not in record
        ]

        extra_fields = [
            field
            for field in observed_fields
            if field not in FIELDS
        ]

        field_order_valid = observed_fields == FIELDS

        if missing_fields or extra_fields:
            schema_issue_rows.append(
                {
                    "Record Index": record_index,
                    "Issue": "Missing or unexpected field names",
                    "Missing Fields": missing_fields,
                    "Extra Fields": extra_fields,
                    "Expected Fields": FIELDS,
                    "Observed Fields": observed_fields,
                    "Field Order Valid": field_order_valid
                }
            )

        comparison_record = {
            field: record.get(field)
            for field in FIELDS
        }

    comparison_record["_extraction_index"] = record_index
    raw_record_rows.append(comparison_record)


extracted_df = pd.DataFrame(raw_record_rows)
schema_issues_df = pd.DataFrame(schema_issue_rows)

record_schema_valid = schema_issues_df.empty

print("Record schema valid:", record_schema_valid)
print("Schema issue count:", len(schema_issues_df))

if not schema_issues_df.empty:
    display(schema_issues_df)

display(extracted_df.head(10))


In [ ]:
# ============================================================
# 7. Input counts and category diagnostics validation
# ============================================================

reference_schema_valid = reference_df.columns.tolist() == FIELDS

reference_record_count_valid = (
    len(reference_df) == EXPECTED_REFERENCE_RECORD_COUNT
)

extraction_record_count_valid = (
    len(extracted_df) == EXPECTED_EXTRACTION_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
)

extraction_category_counts = (
    extracted_df["Category"].value_counts(dropna=False).to_dict()
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

extraction_category_counts_valid = (
    extraction_category_counts == EXPECTED_CATEGORY_COUNTS
)

print("Reference schema valid:", reference_schema_valid)
print("Reference count valid:", reference_record_count_valid)
print("Extraction count valid:", extraction_record_count_valid)
print("Reference category counts valid:", reference_category_counts_valid)
print("Extraction category counts valid:", extraction_category_counts_valid)

if not reference_schema_valid:
    raise ValueError("The D6 reference schema is invalid.")

if not reference_record_count_valid:
    raise ValueError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} reference records, "
        f"found {len(reference_df)}."
    )

if not reference_category_counts_valid:
    raise ValueError(
        "The fixed Stage 1 D6 reference category counts do not match "
        "the expected reference definition."
    )


In [ ]:
# ============================================================
# 8. Null handling and extracted field-type diagnostics
# ============================================================

def is_null(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


type_issue_rows = []
missing_mandatory_rows = []

for row_index, row in extracted_df.iterrows():

    for field in STRING_FIELDS:
        value = row[field]

        if is_null(value):
            missing_mandatory_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field
                }
            )

        elif not isinstance(value, str):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__
                }
            )

    for field in NUMERIC_FIELDS:
        value = row[field]

        if (
            not is_null(value)
            and (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
            )
        ):
            type_issue_rows.append(
                {
                    "Record Index": int(row_index),
                    "Field": field,
                    "Observed Type": type(value).__name__
                }
            )


type_issues_df = pd.DataFrame(type_issue_rows)
missing_mandatory_df = pd.DataFrame(missing_mandatory_rows)

field_types_valid = type_issues_df.empty
mandatory_fields_complete = missing_mandatory_df.empty

print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)
print("Type issues:", len(type_issues_df))
print("Missing mandatory values:", len(missing_mandatory_df))

if not type_issues_df.empty:
    display(type_issues_df)

if not missing_mandatory_df.empty:
    display(missing_mandatory_df)


In [ ]:
# ============================================================
# 9. Controlled comparison normalisation
# ============================================================

def normalise_text(value):
    if is_null(value):
        return None

    text = unicodedata.normalize("NFKC", str(value))

    replacements = {
        "\u00a0": " ",
        "\u2007": " ",
        "\u202f": " ",
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "\u2018": "'",
        "\u2019": "'"
    }

    for source, target in replacements.items():
        text = text.replace(source, target)

    text = re.sub(r"\s+", " ", text)

    return text.strip().casefold()


STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "for", "from", "in",
    "is", "of", "on", "or", "the", "to", "was", "were", "with"
}


def comparison_tokens(value):
    text = normalise_text(value)

    if text is None:
        return set()

    text = re.sub(r"[^a-z0-9]+", " ", text)

    return {
        token
        for token in text.split()
        if token and token not in STOPWORDS
    }


def sequence_similarity(first, second):
    first_text = normalise_text(first)
    second_text = normalise_text(second)

    if first_text is None and second_text is None:
        return 1.0

    if first_text is None or second_text is None:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text
    ).ratio()


def jaccard_similarity(first, second):
    first_tokens = comparison_tokens(first)
    second_tokens = comparison_tokens(second)

    if not first_tokens and not second_tokens:
        return 1.0

    if not first_tokens or not second_tokens:
        return 0.0

    return (
        len(first_tokens & second_tokens)
        / len(first_tokens | second_tokens)
    )


def text_similarity(first, second):
    # Used for record alignment diagnostics only.
    # This is lexical/string similarity, not semantic similarity.
    return max(
        sequence_similarity(first, second),
        jaccard_similarity(first, second)
    )


In [ ]:
# ============================================================
# 10. Controlled D6 equivalence rules
# ============================================================

UNIT_EQUIVALENCE_MAP = {
    "usd billion": "usd billion",
    "usd billions": "usd billion",
    "billion usd": "usd billion",

    "usd millions": "usd millions",
    "usd millions; percent": "usd millions",

    "usd per share": "usd per share",
    "usd per share; percent": "usd per share",

    "percent": "percent",
    "percentage": "percent",

    "million subscribers": "million subscribers",

    "million shares": "million shares",
    "millions of shares": "million shares"
}


BUSINESS_AREA_EQUIVALENCE_MAP = {
    "total": "total",
    "corporate": "corporate",
    "microsoft": "corporate"
}


STATEMENT_EQUIVALENCE_MAP = {
    "quarterly results":
        "quarterly results and business highlights",

    "business highlights":
        "quarterly results and business highlights",

    "quarterly results and business highlights":
        "quarterly results and business highlights",

    "shareholder returns":
        "quarterly results and business highlights"
}


PERIOD_EQUIVALENCE_MAP = {
    "first quarter fiscal year 2024":
        "quarter ended september 30, 2023",

    "first quarter of fiscal year 2024":
        "quarter ended september 30, 2023",

    "quarter ended september 30, 2023":
        "quarter ended september 30, 2023",

    "three months ended september 30":
        "three months ended september 30, 2023",

    "three months ended september 30, 2023":
        "three months ended september 30, 2023",

    "september 30, 2023 and june 30, 2023":
        "september 30, 2023 and june 30, 2023"
}


def canonical_from_map(value, mapping):
    text = normalise_text(value)

    if text is None:
        return None

    return mapping.get(text, text)


def canonical_unit(value):
    return canonical_from_map(
        value,
        UNIT_EQUIVALENCE_MAP
    )


def canonical_business_area(value):
    return canonical_from_map(
        value,
        BUSINESS_AREA_EQUIVALENCE_MAP
    )


def canonical_statement(value):
    return canonical_from_map(
        value,
        STATEMENT_EQUIVALENCE_MAP
    )


def canonical_period(value):
    return canonical_from_map(
        value,
        PERIOD_EQUIVALENCE_MAP
    )


def canonical_metric(value):
    text = normalise_text(value)

    if text is None:
        return None

    text = text.replace(
        "weighted average shares outstanding - basic",
        "basic weighted average shares outstanding"
    )

    text = text.replace(
        "weighted average shares outstanding - diluted",
        "diluted weighted average shares outstanding"
    )

    return text

In [ ]:
# ============================================================
# 11. Controlled Metric + Business Area equivalence
# ============================================================


METRIC_BUSINESS_EQUIVALENCE_RAW = [

    # Microsoft Cloud
    (
        ("Microsoft Cloud revenue", "Microsoft Cloud"),
        ("Revenue", "Microsoft Cloud")
    ),

    # Office Commercial
    (
        (
            "Office Commercial products and cloud services revenue",
            "Office Commercial"
        ),
        (
            "Revenue",
            "Office Commercial products and cloud services"
        )
    ),

    # Office 365 Commercial
    (
        (
            "Office 365 Commercial revenue",
            "Office 365 Commercial"
        ),
        (
            "Revenue growth",
            "Office 365 Commercial"
        )
    ),

    # Office Consumer
    (
        (
            "Office Consumer products and cloud services revenue",
            "Office Consumer"
        ),
        (
            "Revenue",
            "Office Consumer products and cloud services"
        )
    ),

    # LinkedIn
    (
        ("LinkedIn revenue", "LinkedIn"),
        ("Revenue", "LinkedIn")
    ),

    # Dynamics products and cloud services
    (
        (
            "Dynamics products and cloud services revenue",
            "Dynamics"
        ),
        (
            "Revenue",
            "Dynamics products and cloud services"
        )
    ),

    # Dynamics 365
    (
        ("Dynamics 365 revenue", "Dynamics 365"),
        ("Revenue growth", "Dynamics 365")
    ),

    # Server products and cloud services
    (
        (
            "Server products and cloud services revenue",
            "Server products and cloud services"
        ),
        (
            "Revenue",
            "Server products and cloud services"
        )
    ),

    # Azure
    (
        (
            "Azure and other cloud services revenue",
            "Azure and other cloud services"
        ),
        (
            "Revenue growth",
            "Azure and other cloud services"
        )
    ),

    # Windows
    (
        ("Windows revenue", "Windows"),
        ("Revenue", "Windows")
    ),

    # Windows OEM
    (
        ("Windows OEM revenue", "Windows OEM"),
        ("Revenue growth", "Windows OEM")
    ),

    # Windows Commercial
    (
        (
            "Windows Commercial products and cloud services revenue",
            "Windows Commercial"
        ),
        (
            "Revenue growth",
            "Windows Commercial products and cloud services"
        )
    ),

    # Devices
    (
        ("Devices revenue", "Devices"),
        ("Revenue", "Devices")
    ),

    # Xbox
    (
        (
            "Xbox content and services revenue",
            "Xbox content and services"
        ),
        (
            "Revenue",
            "Xbox content and services"
        )
    ),

    # Search and news advertising — narrative
    (
        (
            "Search and news advertising revenue excluding traffic acquisition costs",
            "Search and news advertising"
        ),
        (
            "Revenue excluding traffic acquisition costs",
            "Search and news advertising"
        )
    ),

    # Search and news advertising — reconciliation table
    (
        (
            "Revenue",
            "Search and news advertising excluding traffic acquisition costs"
        ),
        (
            "Revenue excluding traffic acquisition costs",
            "Search and news advertising"
        )
    )
]


def normalise_metric_business_pair(metric, business_area):
    return (
        canonical_metric(metric),
        canonical_business_area(business_area)
    )


METRIC_BUSINESS_EQUIVALENCE = {
    (
        normalise_metric_business_pair(
            reference_metric,
            reference_business
        ),
        normalise_metric_business_pair(
            extracted_metric,
            extracted_business
        )
    )
    for (
        (reference_metric, reference_business),
        (extracted_metric, extracted_business)
    )
    in METRIC_BUSINESS_EQUIVALENCE_RAW
}


def is_metric_business_pair_equivalent(
    reference_metric,
    reference_business,
    extracted_metric,
    extracted_business
):
    reference_pair = normalise_metric_business_pair(
        reference_metric,
        reference_business
    )

    extracted_pair = normalise_metric_business_pair(
        extracted_metric,
        extracted_business
    )

    return (
        reference_pair,
        extracted_pair
    ) in METRIC_BUSINESS_EQUIVALENCE

In [ ]:
# ============================================================
# 12. Numeric and exact-value comparison
# ============================================================

def numeric_value(value):
    if is_null(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        text = (
            value.strip()
            .replace(",", "")
            .replace("−", "-")
            .replace("–", "-")
        )

        if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
            return float(text)

    return None


def values_match(reference_value, extracted_value, tolerance=1e-9):

    # Both absent = agreement
    if is_null(reference_value) and is_null(extracted_value):
        return True

    # Only one absent = discrepancy
    if is_null(reference_value) or is_null(extracted_value):
        return False

    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if reference_number is not None and extracted_number is not None:
        return math.isclose(
            reference_number,
            extracted_number,
            rel_tol=tolerance,
            abs_tol=tolerance
        )

    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


In [ ]:
# ============================================================
# 13. Comparison copies and blocking keys preparation
# ============================================================

reference_comparison_df = reference_df.copy(deep=True)
extracted_comparison_df = extracted_df.copy(deep=True)

reference_comparison_df["_reference_index"] = range(
    len(reference_comparison_df)
)

for dataframe in [
    reference_comparison_df,
    extracted_comparison_df
]:
    dataframe["_block_category"] = dataframe["Category"].map(normalise_text)
    dataframe["_block_source"] = dataframe["Source Location"].map(normalise_text)

    dataframe["_matching_block"] = list(
        zip(
            dataframe["_block_category"],
            dataframe["_block_source"]
        )
    )

print("Comparison copies prepared.")
print("Blocking fields:", BLOCK_FIELDS)


In [ ]:
# ============================================================
# 14. Deterministic identity-based record-matching score
# ============================================================

def matching_score(reference_row, extracted_row):

    metric_score = text_similarity(
        canonical_metric(reference_row["Metric"]),
        canonical_metric(extracted_row["Metric"])
    )

    business_score = (
        1.0
        if canonical_business_area(reference_row["Business Area"])
        == canonical_business_area(extracted_row["Business Area"])
        else text_similarity(
            reference_row["Business Area"],
            extracted_row["Business Area"]
        )
    )

    statement_score = (
        1.0
        if canonical_statement(reference_row["Statement or Section"])
        == canonical_statement(extracted_row["Statement or Section"])
        else text_similarity(
            reference_row["Statement or Section"],
            extracted_row["Statement or Section"]
        )
    )

    period_score = (
        1.0
        if canonical_period(reference_row["Reporting Period"])
        == canonical_period(extracted_row["Reporting Period"])
        else text_similarity(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"]
        )
    )

    total_score = (
        0.55 * metric_score
        + 0.25 * business_score
        + 0.15 * statement_score
        + 0.05 * period_score
    )

    return {
        "total": total_score,
        "metric": metric_score,
        "business_area": business_score,
        "statement": statement_score,
        "reporting_period": period_score
    }


In [ ]:
# ============================================================
# 15. One-to-one record alignment
# ============================================================

all_blocks = sorted(
    set(reference_comparison_df["_matching_block"])
    | set(extracted_comparison_df["_matching_block"]),
    key=str
)

matched_pairs = []

unmatched_reference_indices = set(
    reference_comparison_df["_reference_index"].tolist()
)

unmatched_extraction_indices = set(
    extracted_comparison_df["_extraction_index"].tolist()
)


for block in all_blocks:

    reference_block = reference_comparison_df.loc[
        reference_comparison_df["_matching_block"] == block
    ]

    extraction_block = extracted_comparison_df.loc[
        extracted_comparison_df["_matching_block"] == block
    ]

    if reference_block.empty or extraction_block.empty:
        continue

    reference_rows = [
        row
        for _, row in reference_block.iterrows()
    ]

    extraction_rows = [
        row
        for _, row in extraction_block.iterrows()
    ]

    score_matrix = []
    details_matrix = []

    for reference_row in reference_rows:

        score_row = []
        details_row = []

        for extracted_row in extraction_rows:
            details = matching_score(
                reference_row,
                extracted_row
            )

            score_row.append(details["total"])
            details_row.append(details)

        score_matrix.append(score_row)
        details_matrix.append(details_row)

    cost_matrix = [
        [1.0 - score for score in row]
        for row in score_matrix
    ]

    row_positions, column_positions = linear_sum_assignment(
        cost_matrix
    )

    for row_position, column_position in zip(
        row_positions,
        column_positions
    ):

        details = details_matrix[
            row_position
        ][
            column_position
        ]

        if details["total"] < MATCH_SCORE_THRESHOLD:
            continue

        reference_index = int(
            reference_rows[
                row_position
            ][
                "_reference_index"
            ]
        )

        extraction_index = int(
            extraction_rows[
                column_position
            ][
                "_extraction_index"
            ]
        )

        matched_pairs.append(
            {
                "reference_index": reference_index,
                "extraction_index": extraction_index,
                "matching_score": details["total"],
                "metric_matching_score": details["metric"],
                "business_area_matching_score":
                    details["business_area"],
                "statement_matching_score": details["statement"],
                "reporting_period_matching_score":
                    details["reporting_period"]
            }
        )

        unmatched_reference_indices.discard(
            reference_index
        )

        unmatched_extraction_indices.discard(
            extraction_index
        )


aligned_record_count = len(matched_pairs)
missing_record_count = len(unmatched_reference_indices)
unsupported_record_count = len(unmatched_extraction_indices)

print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print(
    "Unsupported/unmatched extracted records:",
    unsupported_record_count
)


In [ ]:
# ============================================================
# 16. Missing and unsupported/unmatched record tables
# ============================================================

missing_records_df = reference_comparison_df.loc[
    reference_comparison_df["_reference_index"].isin(
        unmatched_reference_indices
    ),
    ["_reference_index"] + FIELDS
].copy()

unsupported_records_df = extracted_comparison_df.loc[
    extracted_comparison_df["_extraction_index"].isin(
        unmatched_extraction_indices
    ),
    ["_extraction_index"] + FIELDS
].copy()

print("Missing:", len(missing_records_df))
print("Unsupported/unmatched:", len(unsupported_records_df))

if not missing_records_df.empty:
    display(missing_records_df)

if not unsupported_records_df.empty:
    display(unsupported_records_df)


In [ ]:
# ============================================================
# 17. Field-level comparison of aligned records
# ============================================================

def exact_text_match(first, second):
    return normalise_text(first) == normalise_text(second)


def mapped_text_match(first, second, canonical_function):
    return canonical_function(first) == canonical_function(second)


comparison_rows = []

for pair in matched_pairs:

    reference_row = reference_comparison_df.loc[
        reference_comparison_df["_reference_index"]
        == pair["reference_index"]
    ].iloc[0]

    extracted_row = extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"]
        == pair["extraction_index"]
    ].iloc[0]

    category_match = exact_text_match(
        reference_row["Category"],
        extracted_row["Category"]
    )

    statement_match = mapped_text_match(
        reference_row["Statement or Section"],
        extracted_row["Statement or Section"],
        canonical_statement
    )

    metric_similarity = text_similarity(
        canonical_metric(reference_row["Metric"]),
        canonical_metric(extracted_row["Metric"])
    )

    metric_business_pair_rule_applied = (
        is_metric_business_pair_equivalent(
            reference_row["Metric"],
            reference_row["Business Area"],
            extracted_row["Metric"],
            extracted_row["Business Area"]
        )
    )


    metric_match = (
        canonical_metric(reference_row["Metric"])
        == canonical_metric(extracted_row["Metric"])
        or metric_business_pair_rule_applied
    )


    business_area_match = (
        canonical_business_area(
            reference_row["Business Area"]
        )
        == canonical_business_area(
            extracted_row["Business Area"]
        )
        or metric_business_pair_rule_applied
    )

    value_2023_match = values_match(
        reference_row["Value 2023"],
        extracted_row["Value 2023"]
    )

    value_2022_match = values_match(
        reference_row["Value 2022"],
        extracted_row["Value 2022"]
    )

    gaap_change_match = values_match(
        reference_row["GAAP YoY Change"],
        extracted_row["GAAP YoY Change"]
    )

    constant_currency_impact_match = values_match(
        reference_row["Constant Currency Impact"],
        extracted_row["Constant Currency Impact"]
    )

    constant_currency_change_match = values_match(
        reference_row["Constant Currency YoY Change"],
        extracted_row["Constant Currency YoY Change"]
    )

    unit_match = mapped_text_match(
        reference_row["Unit"],
        extracted_row["Unit"],
        canonical_unit
    )

    reporting_period_match = mapped_text_match(
        reference_row["Reporting Period"],
        extracted_row["Reporting Period"],
        canonical_period
    )

    source_location_match = exact_text_match(
        reference_row["Source Location"],
        extracted_row["Source Location"]
    )

    field_matches = {
        "Category": category_match,
        "Statement or Section": statement_match,
        "Metric": metric_match,
        "Business Area": business_area_match,
        "Value 2023": value_2023_match,
        "Value 2022": value_2022_match,
        "GAAP YoY Change": gaap_change_match,
        "Constant Currency Impact":
            constant_currency_impact_match,
        "Constant Currency YoY Change":
            constant_currency_change_match,
        "Unit": unit_match,
        "Reporting Period": reporting_period_match,
        "Source Location": source_location_match
    }

    all_primary_fields_match = all(
        field_matches[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    identity_fields_match = all(
        field_matches[field]
        for field in ALIGNMENT_IDENTITY_FIELDS
    )

    identity_label_difference = (
        all_primary_fields_match
        and not identity_fields_match
    )

    all_mismatched_fields = [
        field
        for field, match in field_matches.items()
        if not match
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    output_row = {
        "Reference Index": pair["reference_index"],
        "Extraction Index": pair["extraction_index"],
        "Matching Score": pair["matching_score"],

        "Metric Matching Score":
            pair["metric_matching_score"],

        "Business Area Matching Score":
            pair["business_area_matching_score"],

        "Statement Matching Score":
            pair["statement_matching_score"],

        "Reporting Period Matching Score":
            pair["reporting_period_matching_score"],

        "Category":
            reference_row["Category"],

        "Reference Metric":
            reference_row["Metric"],

        "Extracted Metric":
            extracted_row["Metric"],

        "Metric Lexical Similarity":
            metric_similarity,

        "Metric-Business Pair Rule Applied":
            bool(metric_business_pair_rule_applied),

        "identity_fields_match":
            bool(identity_fields_match),

        "identity_label_difference":
            bool(identity_label_difference),

        "Fully Correct":
            bool(all_primary_fields_match),

        "all_mismatched_fields":
            ", ".join(all_mismatched_fields),

        "primary_mismatched_fields":
            ", ".join(primary_mismatched_fields)
    }



    for field in FIELDS:
        output_row[f"Reference {field}"] = reference_row[field]
        output_row[f"Extracted {field}"] = extracted_row[field]
        output_row[f"{field} Match"] = bool(field_matches[field])

    comparison_rows.append(output_row)


comparison_df = pd.DataFrame(comparison_rows)

print("Compared aligned records:", len(comparison_df))
print(
    "Fully correct:",
    int(comparison_df["Fully Correct"].sum())
)

display(comparison_df.head(10))


In [ ]:
# ============================================================
# 18. Fully-correct and discrepant aligned records
# ============================================================

fully_correct_records_df = comparison_df.loc[
    comparison_df["Fully Correct"] == True
].copy()

discrepant_records_df = comparison_df.loc[
    comparison_df["Fully Correct"] == False
].copy()

print("Fully correct aligned records:", len(fully_correct_records_df))
print("Discrepant aligned records:", len(discrepant_records_df))

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Category",
                "Reference Metric",
                "Extracted Metric",
                "primary_mismatched_fields"
            ]
        ].head(30)
    )


In [ ]:
# ============================================================
# 19. Field discrepancy table
# ============================================================

field_discrepancy_rows = []

for _, row in comparison_df.iterrows():

    for field in PRIMARY_CORRECTNESS_FIELDS:

        if not bool(row[f"{field} Match"]):
            field_discrepancy_rows.append(
                {
                    "Reference Index":
                        int(row["Reference Index"]),
                    "Extraction Index":
                        int(row["Extraction Index"]),
                    "Category":
                        row["Category"],
                    "Reference Metric":
                        row["Reference Metric"],
                    "Extracted Metric":
                        row["Extracted Metric"],
                    "Field":
                        field,
                    "Reference Value":
                        row[f"Reference {field}"],
                    "Extracted Value":
                        row[f"Extracted {field}"]
                }
            )


field_discrepancies_df = pd.DataFrame(
    field_discrepancy_rows
)

print(
    "Field discrepancy count:",
    len(field_discrepancies_df)
)

if not field_discrepancies_df.empty:
    display(field_discrepancies_df)


In [ ]:
# ============================================================
# 20. Field-level accuracy among aligned records
# ============================================================

field_accuracy_rows = []

for field in PRIMARY_CORRECTNESS_FIELDS:

    correct_count = int(
        comparison_df[f"{field} Match"].sum()
    )

    aligned_count = len(comparison_df)

    field_accuracy_rows.append(
        {
            "Field": field,
            "Correct Records": correct_count,
            "Aligned Records": aligned_count,
            "Accuracy": (
                correct_count / aligned_count
                if aligned_count > 0
                else None
            )
        }
    )


field_accuracy_df = pd.DataFrame(
    field_accuracy_rows
)

field_accuracy_dictionary = {
    row["Field"]:
        (
            float(row["Accuracy"])
            if pd.notna(row["Accuracy"])
            else None
        )
    for _, row in field_accuracy_df.iterrows()
}

display(field_accuracy_df)


In [ ]:
# ============================================================
# 21. Record-level metrics
# ============================================================

fully_correct_record_count = int(
    comparison_df["Fully Correct"].sum()
)

discrepant_record_count = (
    aligned_record_count
    - fully_correct_record_count
)

completeness = (
    aligned_record_count / len(reference_df)
    if len(reference_df) > 0
    else 0.0
)

missing_rate = (
    missing_record_count / len(reference_df)
    if len(reference_df) > 0
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count / len(extracted_df)
    if len(extracted_df) > 0
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count / len(reference_df)
    if len(reference_df) > 0
    else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if (
        record_precision_exact + record_recall_exact
        > 0
    )
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / len(extracted_df)
    if len(extracted_df) > 0
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count > 0
    else 0.0
)

field_accuracy = (
    sum(
        int(
            comparison_df[
                f"{field} Match"
            ].sum()
        )
        for field in PRIMARY_CORRECTNESS_FIELDS
    )
    / (
        len(comparison_df)
        * len(PRIMARY_CORRECTNESS_FIELDS)
    )
    if len(comparison_df) > 0
    else None
)

schema_error_rate = (
    len(schema_issues_df) / len(extracted_df)
    if len(extracted_df) > 0
    else 0.0
)

print("Fully correct records:", fully_correct_record_count)
print("Discrepant records:", discrepant_record_count)
print("Completeness:", completeness)
print("Exact record precision:", record_precision_exact)
print("Exact record recall:", record_recall_exact)
print("Exact record F1:", record_f1_exact)
print("Missing rate:", missing_rate)
print("Unsupported rate:", unsupported_rate)
print(
    "Discrepancy rate among aligned:",
    discrepancy_rate_among_aligned
)
print(
    "Field accuracy:",
    field_accuracy
)
print("Schema error rate:", schema_error_rate)


In [ ]:
# ============================================================
# 22. Category-level metrics
# ============================================================

category_metric_rows = []

for category, expected_count in EXPECTED_CATEGORY_COUNTS.items():

    category_comparison = comparison_df.loc[
        comparison_df["Category"] == category
    ]

    extracted_category_count = int(
        (
            extracted_df["Category"]
            == category
        ).sum()
    )

    aligned_count = len(category_comparison)

    fully_correct_count = int(
        category_comparison[
            "Fully Correct"
        ].sum()
    )

    discrepant_count = (
        aligned_count - fully_correct_count
    )

    category_completeness = (
        aligned_count / expected_count
        if expected_count > 0
        else None
    )

    category_precision_exact = (
        fully_correct_count
        / extracted_category_count
        if extracted_category_count > 0
        else 0.0
    )

    category_recall_exact = (
        fully_correct_count
        / expected_count
        if expected_count > 0
        else 0.0
    )

    category_f1_exact = (
        2
        * category_precision_exact
        * category_recall_exact
        / (
            category_precision_exact
            + category_recall_exact
        )
        if (
            category_precision_exact
            + category_recall_exact
            > 0
        )
        else 0.0
    )

    category_metric_rows.append(
        {
            "Category": category,
            "Expected Records": expected_count,
            "Extracted Records": extracted_category_count,
            "Aligned Records": aligned_count,
            "Fully Correct Records": fully_correct_count,
            "Discrepant Records": discrepant_count,
            "Completeness": category_completeness,
            "Record Precision Exact":
                category_precision_exact,
            "Record Recall Exact":
                category_recall_exact,
            "Record F1 Exact":
                category_f1_exact
        }
    )


category_metrics_df = pd.DataFrame(
    category_metric_rows
)

display(category_metrics_df)


In [ ]:
# ============================================================
# 23. Schema validity definition independently from completeness
# ============================================================

schema_diagnostics = {
    "valid_json": bool(valid_json),
    "top_level_object_valid": bool(top_level_object_valid),
    "document_id_correct": bool(document_id_correct),
    "branch_correct": bool(branch_correct),
    "records_is_list": bool(records_is_list),
    "record_schema_valid": bool(record_schema_valid),
    "field_types_valid": bool(field_types_valid),
    "records_with_structure_issues":
        int(len(schema_issues_df)),
    "records_with_type_issues":
        int(len(type_issues_df)),
    "local_record_schema_valid":
        bool(record_schema_valid),
    "local_field_types_valid":
        bool(field_types_valid),
    "structurally_evaluable":
        bool(branch_structurally_evaluable)
}

schema_validity = bool(
    branch_structurally_evaluable
)

schema_diagnostics["schema_validity"] = bool(
    schema_validity
)

print(
    json.dumps(
        schema_diagnostics,
        indent=2
    )
)

print(
    "\nContent diagnostics kept separate from schema:"
)
print(
    "Extraction count valid:",
    extraction_record_count_valid
)
print(
    "Extraction category counts valid:",
    extraction_category_counts_valid
)
print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)


In [ ]:
# ============================================================
# 24. Validation summary table
# ============================================================

validation_summary_df = pd.DataFrame(
    [
        {
            "Metric": "Reference records",
            "Value": len(reference_df)
        },
        {
            "Metric": "Extracted records",
            "Value": len(extracted_df)
        },
        {
            "Metric": "Aligned records",
            "Value": aligned_record_count
        },
        {
            "Metric": "Fully correct records",
            "Value": fully_correct_record_count
        },
        {
            "Metric": "Discrepant records",
            "Value": discrepant_record_count
        },
        {
            "Metric": "Missing records",
            "Value": missing_record_count
        },
        {
            "Metric": "Unsupported extracted records",
            "Value": unsupported_record_count
        },
        {
            "Metric": "Completeness",
            "Value": completeness
        },
        {
            "Metric": "Record precision exact",
            "Value": record_precision_exact
        },
        {
            "Metric": "Record recall exact",
            "Value": record_recall_exact
        },
        {
            "Metric": "Record F1 exact",
            "Value": record_f1_exact
        },
        {
            "Metric": "field accuracy",
            "Value": field_accuracy
        },
        {
            "Metric": "Schema validity",
            "Value": schema_validity
        },
        {
            "Metric": "Schema issue count",
            "Value": len(schema_issues_df)
        }
    ]
)

display(validation_summary_df)


In [ ]:
# ============================================================
# 25. Reproducible validation metrics
# ============================================================

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records":
            int(row["Expected Records"]),
        "extracted_records":
            int(row["Extracted Records"]),
        "aligned_records":
            int(row["Aligned Records"]),
        "fully_correct_records":
            int(row["Fully Correct Records"]),
        "discrepant_records":
            int(row["Discrepant Records"]),
        "completeness":
            float(row["Completeness"]),
        "record_precision_exact":
            float(row["Record Precision Exact"]),
        "record_recall_exact":
            float(row["Record Recall Exact"]),
        "record_f1_exact":
            float(row["Record F1 Exact"])
    }
    for _, row in category_metrics_df.iterrows()
}


VALIDATION_METRICS = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "input_representation": INPUT_REPRESENTATION,
    "alignment_identity_fields": ALIGNMENT_IDENTITY_FIELDS,
    "primary_correctness_fields": PRIMARY_CORRECTNESS_FIELDS,

    "reference_records":
        int(len(reference_df)),
    "extracted_records":
        int(len(extracted_df)),
    "aligned_records":
        int(aligned_record_count),
    "fully_correct_records":
        int(fully_correct_record_count),
    "discrepant_records":
        int(discrepant_record_count),
    "missing_records":
        int(missing_record_count),
    "unsupported_extracted_records":
        int(unsupported_record_count),

    "completeness":
        round(completeness, 4),
    "missing_rate":
        round(missing_rate, 4),
    "record_precision_exact":
        round(record_precision_exact, 4),
    "record_recall_exact":
        round(record_recall_exact, 4),
    "record_f1_exact":
        round(record_f1_exact, 4),
    "unsupported_rate":
        round(unsupported_rate, 4),
    "discrepancy_rate_among_aligned":
        round(
            discrepancy_rate_among_aligned,
            4
        ),

    "field_accuracy":
        (
            round(
                field_accuracy,
                4
            )
            if field_accuracy
            is not None
            else None
        ),

    "field_accuracy_among_aligned": {
        field:
            (
                round(value, 4)
                if value is not None
                else None
            )
        for field, value
        in field_accuracy_dictionary.items()
    },

    "schema_validity":
        bool(schema_validity),
    "schema_diagnostics":
        schema_diagnostics,

    "content_diagnostics": {
        "reference_record_count_valid":
            bool(reference_record_count_valid),
        "reference_category_counts_valid":
            bool(reference_category_counts_valid),
        "extraction_record_count_valid":
            bool(extraction_record_count_valid),
        "extraction_category_counts_valid":
            bool(extraction_category_counts_valid),
        "mandatory_fields_complete":
            bool(mandatory_fields_complete)
    },

    "matching_rules": {
        "blocking_fields":
            BLOCK_FIELDS,
        "one_to_one_assignment":
            "Hungarian linear-sum assignment",
        "matching_score_threshold":
            MATCH_SCORE_THRESHOLD,
        "matching_score_weights": {
            "metric": 0.55,
            "business_area": 0.25,
            "statement_or_section": 0.15,
            "reporting_period": 0.05
        },
        "value_2023_used_for_alignment":
            False,
        "value_2022_used_for_alignment":
            False,
        "unit_used_for_alignment":
            False,
        "gaap_change_used_for_alignment":
            False,
        "constant_currency_fields_used_for_alignment":
            False
    },

    "comparison_rules": {
        "raw_extraction_modified":
            False,
        "manual_correction_applied":
            False,
        "unexpected_fields_copied_to_expected_fields":
            False,
        "comparison_normalisation_scope":
            "Comparison copies only",
        "null_comparison":
            "None and pandas NaN treated as equivalent absence",
        "numeric_comparison":
            "Exact numeric equality after deterministic parsing",
        "metric":
            (
                "Normalised exact matching supplemented by "
                "predefined source-grounded Metric + Business Area "
                "pair equivalence where observation identity is "
                "distributed across both fields"
            ),

        "metric_business_pair_equivalence":
            (
                "Controlled document-level equivalence for verified "
                "alternative partitions of the same source observation"
            ),

        "metric_business_equivalence_rules_frozen_across_branches":
            True,

        "metric_lexical_similarity":
            (
                "Diagnostic only; lexical similarity is not used "
                "to determine field correctness"
            ),

        "business_area":
            (
                "Controlled source-grounded equivalence map, "
                "supplemented by predefined Metric + Business Area "
                "pair equivalence"
            ),
        "statement_or_section":
            (
                "Controlled source-grounded equivalence map; "
                "Quarterly results, Business Highlights and "
                "Shareholder returns are treated as equivalent "
                "within the predefined D6 source scope"
            ),

        "unit":
            (
                "Controlled source-grounded equivalence map; "
                "equivalent ordering variants are canonicalised, "
                "and reconciliation compound units are reduced to "
                "the base value unit because percentage changes are "
                "represented in dedicated schema fields"
            ),

        "reporting_period":
            (
                "Controlled source-grounded canonical equivalence "
                "for fiscal-quarter and quarter-end formulations"
            ),

        "source_location":
            "Normalised exact correctness after alignment",
        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,
        "d6_equivalence_rules_frozen_across_branches":
            True,
    },

    "category_metrics":
        category_metrics_dictionary,

    "normalisation_note":
        (
            "Deterministic normalisation was applied only to "
            "comparison copies; the preserved Branch A extraction "
            "was not modified."
        ),

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,
        "reference_sha256":
            REFERENCE_SHA256,
        "combined_extraction_file":
            EXTRACTION_PATH.name,
        "combined_extraction_sha256":
            EXTRACTION_SHA256,
        "technical_diagnostics_file":
            TECHNICAL_DIAGNOSTICS_PATH.name,
        "technical_diagnostics_sha256":
            TECHNICAL_DIAGNOSTICS_SHA256,
        "branch_A_structurally_evaluable":
            branch_structurally_evaluable
    },

}


print(
    json.dumps(
        VALIDATION_METRICS,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 26. Validation metadata and conclusion
# ============================================================

VALIDATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "reference_file":
        REFERENCE_PATH.name,
    "reference_file_sha256":
        REFERENCE_SHA256,

    "extraction_file":
        EXTRACTION_PATH.name,
    "extraction_file_sha256":
        EXTRACTION_SHA256,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,
    "technical_diagnostics_file_sha256":
        TECHNICAL_DIAGNOSTICS_SHA256,

    "validation_type":
        (
            "Deterministic comparison against the fixed "
            "D6 Stage 1 reference dataset"
        ),

    "raw_extraction_modified":
        False,
    "manual_correction_applied":
        False,
    "schema_errors_preserved":
        True,
    "comparison_normalisation_scope":
        "Comparison copies only",

    "matching_outcome_values_used":
        False,

    "automatic_unmatched_label":
        "unsupported/unmatched; not automatically hallucinated",

    "notes":
        (
            "Validation A compares the fixed 147-record D6 Stage 1 "
            "reference dataset with the untouched combined Branch A "
            "extraction generated from five predefined source-section "
            "runs. The five-part extraction protocol belongs to "
            "Branch A execution; validation is performed on the "
            "combined parsed extraction. Missing or malformed fields "
            "are preserved as observed and are not repaired."
        )
}


validation_status = (
    "Completed without discrepancies"
    if (
        fully_correct_record_count
        == EXPECTED_REFERENCE_RECORD_COUNT
        and missing_record_count == 0
        and unsupported_record_count == 0
        and schema_validity
    )
    else "Completed with discrepancies"
)


VALIDATION_CONCLUSION = {
    "document_id":
        DOCUMENT_ID,
    "branch":
        BRANCH,
    "branch_name":
        BRANCH_NAME,

    "validation_status":
        validation_status,

    "reference_records":
        int(len(reference_df)),
    "extracted_records":
        int(len(extracted_df)),
    "aligned_records":
        int(aligned_record_count),
    "fully_correct_records":
        int(fully_correct_record_count),
    "discrepant_records":
        int(discrepant_record_count),
    "missing_records":
        int(missing_record_count),
    "unsupported_extracted_records":
        int(unsupported_record_count),

    "completeness":
        round(completeness, 4),
    "record_precision_exact":
        round(record_precision_exact, 4),
    "record_recall_exact":
        round(record_recall_exact, 4),
    "record_f1_exact":
        round(record_f1_exact, 4),

    "field_accuracy":
        (
            round(
                field_accuracy,
                4
            )
            if field_accuracy
            is not None
            else None
        ),

    "schema_valid":
        bool(schema_validity),
    "schema_issue_count":
        int(len(schema_issues_df)),

    "notes":
        (
            "The raw Branch A extraction was preserved unchanged. "
            "Schema validity, content completeness, field-level "
            "agreement, missing records and unsupported/unmatched "
            "records are reported separately."
        )
}


print(
    json.dumps(
        VALIDATION_CONCLUSION,
        ensure_ascii=False,
        indent=2
    )
)


In [ ]:
# ============================================================
# 27. Define output paths
# ============================================================

DETAILED_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_validation_detailed.csv"
)

FULLY_CORRECT_RECORDS_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_fully_correct_records.csv"
)

DISCREPANT_RECORDS_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_discrepant_records.csv"
)

MISSING_RECORDS_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_missing_records.csv"
)

UNSUPPORTED_RECORDS_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_unsupported_records.csv"
)

SCHEMA_ISSUES_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_schema_issues.csv"
)

TYPE_ISSUES_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_type_issues.csv"
)

FIELD_DISCREPANCIES_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_field_validation.csv"
)

FIELD_ACCURACY_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_field_error_summary.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_category_metrics.csv"
)

VALIDATION_SUMMARY_CSV_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_validation_summary.csv"
)

VALIDATION_SUMMARY_JSON_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_validation_summary.json"
)

VALIDATION_METADATA_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_validation_metadata.json"
)

VALIDATION_CONCLUSION_PATH = (
    OUTPUT_DIR
    / "D6_branch_A_validation_conclusion.json"
)


In [ ]:
# ============================================================
# 28. Export Validation A outputs
# ============================================================

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

discrepant_records_df.to_csv(
    DISCREPANT_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_records_df.to_csv(
    MISSING_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_records_df.to_csv(
    UNSUPPORTED_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

schema_issues_df.to_csv(
    SCHEMA_ISSUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

type_issues_df.to_csv(
    TYPE_ISSUES_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_discrepancies_df.to_csv(
    FIELD_DISCREPANCIES_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_accuracy_df.to_csv(
    FIELD_ACCURACY_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

validation_summary_df.to_csv(
    VALIDATION_SUMMARY_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

for output_path, content in [
    (
        VALIDATION_SUMMARY_JSON_PATH,
        VALIDATION_METRICS
    ),
    (
        VALIDATION_METADATA_PATH,
        VALIDATION_METADATA
    ),
    (
        VALIDATION_CONCLUSION_PATH,
        VALIDATION_CONCLUSION
    )
]:
    output_path.write_text(
        json.dumps(
            content,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

print("D6 Validation A outputs exported.")


In [ ]:
# ============================================================
# 29. Final validation consistency checks
# ============================================================

if not reference_schema_valid:
    raise AssertionError(
        "Reference schema validation failed."
    )

if not reference_record_count_valid:
    raise AssertionError(
        "Reference record-count validation failed."
    )

if not reference_category_counts_valid:
    raise AssertionError(
        "Reference category-count validation failed."
    )

if not field_types_valid:
    raise AssertionError(
        "The extracted comparison fields contain invalid data types."
    )

if (
    aligned_record_count
    + missing_record_count
    != len(reference_df)
):
    raise AssertionError(
        "Reference-record accounting is inconsistent."
    )

if (
    aligned_record_count
    + unsupported_record_count
    != len(extracted_df)
):
    raise AssertionError(
        "Extraction-record accounting is inconsistent."
    )

print("Validation status:", validation_status)
print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_df))
print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print(
    "Unsupported/unmatched extracted records:",
    unsupported_record_count
)
print("Schema issues:", len(schema_issues_df))
print("Schema valid:", schema_validity)
print(
    "Fully correct records:",
    fully_correct_record_count
)
print(
    "Discrepant records:",
    discrepant_record_count
)
print(
    "Field accuracy:",
    field_accuracy
)
print("Exact record precision:", record_precision_exact)
print("Exact record recall:", record_recall_exact)
print("Exact record F1:", record_f1_exact)

print("\nD6 Validation A completed successfully.")


In [ ]:
# ============================================================
# 30. List generated outputs
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_RECORDS_PATH,
    DISCREPANT_RECORDS_PATH,
    MISSING_RECORDS_PATH,
    UNSUPPORTED_RECORDS_PATH,
    SCHEMA_ISSUES_PATH,
    TYPE_ISSUES_PATH,
    FIELD_DISCREPANCIES_PATH,
    FIELD_ACCURACY_PATH,
    CATEGORY_METRICS_PATH,
    VALIDATION_SUMMARY_CSV_PATH,
    VALIDATION_SUMMARY_JSON_PATH,
    VALIDATION_METADATA_PATH,
    VALIDATION_CONCLUSION_PATH
]

print("Generated D6 Validation A files:\n")

for output_path in GENERATED_OUTPUTS:
    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )
